In [33]:
import pandas as pd
import numpy as np
import pymc as pm

from gurobipy import *
import pytensor.tensor as pt

In [4]:
managers_0 = pd.read_csv('../content/managers_0.csv')
managers_1 = pd.read_csv('../content/managers_1.csv')
managers_2 = pd.read_csv('../content/managers_2.csv')
managers_3 = pd.read_csv('../content/managers_3.csv')
managers_4 = pd.read_csv('../content/managers_4.csv')
managers_5 = pd.read_csv('../content/managers_5.csv')
managers_6 = pd.read_csv('../content/managers_6.csv')
managers_7 = pd.read_csv('../content/managers_7.csv')
managers_8 = pd.read_csv('../content/managers_8.csv')
managers_9 = pd.read_csv('../content/managers_9.csv')
managers_10 = pd.read_csv('../content/managers_10.csv')
managers_11 = pd.read_csv('../content/managers_11.csv')
all_managers = pd.concat([managers_0, managers_1, managers_2, managers_3, managers_4, managers_5,managers_6, managers_7, managers_8, managers_9, managers_10, managers_11], axis=0)
all_managers.to_csv('../content/all_managers.csv', index=False)

In [6]:
import ast
all_managers = pd.read_csv('../content/all_managers.csv')
all_managers = all_managers.dropna()
y = set()
for x in all_managers['picks']:
    x_list = set(eval(x))
    # print(x_list
    y = y.union(set(ast.literal_eval(x)))

y

{5,
 6,
 8,
 10,
 12,
 13,
 14,
 17,
 19,
 20,
 25,
 26,
 27,
 28,
 29,
 31,
 32,
 33,
 34,
 36,
 37,
 42,
 43,
 44,
 48,
 49,
 50,
 52,
 53,
 58,
 60,
 69,
 72,
 77,
 80,
 82,
 83,
 84,
 85,
 88,
 89,
 90,
 92,
 96,
 101,
 103,
 104,
 106,
 107,
 108,
 109,
 110,
 112,
 113,
 114,
 116,
 117,
 119,
 120,
 121,
 122,
 126,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 139,
 140,
 141,
 143,
 144,
 145,
 148,
 150,
 151,
 152,
 154,
 156,
 157,
 160,
 161,
 166,
 168,
 178,
 181,
 182,
 187,
 193,
 194,
 195,
 196,
 197,
 198,
 199,
 200,
 202,
 203,
 204,
 205,
 206,
 208,
 211,
 212,
 216,
 217,
 220,
 221,
 222,
 225,
 226,
 228,
 230,
 232,
 234,
 236,
 238,
 242,
 245,
 246,
 249,
 255,
 256,
 257,
 258,
 260,
 262,
 263,
 265,
 266,
 267,
 271,
 275,
 276,
 279,
 280,
 281,
 282,
 283,
 287,
 289,
 290,
 291,
 293,
 294,
 295,
 297,
 298,
 301,
 302,
 303,
 304,
 307,
 308,
 309,
 311,
 313,
 314,
 315,
 316,
 318,
 321,
 326,
 328,
 340,
 341,
 342,
 343,
 344,
 346,
 349,
 350,
 35

In [ ]:
managers = all_managers[all_managers['picks'].isna() == False]
managers

In [ ]:
# Extract picks from managers dataframe
picks_series = managers['picks']

# Convert picks to a list of tuples
picks_list = picks_series.apply(eval).tolist()

# Flatten the list of tuples and extract unique picks
players_picks = list(set([pick for sublist in picks_list for pick in sublist]))
players_picks

## FPL


In [ ]:

import requests

# Define the API endpoint
url = 'https://fantasy.premierleague.com/api/bootstrap-static/'

# Send a GET request to the API
response = requests.get(url)
player_data = None
# Check if the request was successful
if response.status_code == 200:
    data = response.json()  # Parse the JSON data
    players = data['elements']  # Extract the list of players

    player_data = [
        {\"id\": player[\"id\"], \"name\": f\"{player['first_name']} {player['second_name']}\", \"price\": player[\"now_cost\"] / 10, \"position\": player[\"element_type\"]}
        for player in players
    ]

else:
    print('Failed to retrieve data')

fpl_players = pd.DataFrame(player_data)
fpl_players

players = fpl_players['id'].values.tolist()

In [ ]:
pd.read_csv('../FPL predictors/with new features/predicted.csv')['player_team'].isnull().sum()

In [ ]:
pred = pd.read_csv('../FPL predictors/with new features/predicted.csv')

pred_pts = pred[['xPts', 'value', 'fpl_id']].copy()


pred_pts['fpl_id'] = pred_pts['fpl_id'].astype(int)

# divide the value by 10
pred_pts.loc[:, 'value'] = pred_pts['value'] * 10
# pred_pts = pred_pts.apply(lambda x: x/10 for  x in players else None, axis=1).dropna()
pred_players = pred_pts['fpl_id'].tolist()
pred_pts[pred_pts['fpl_id'] ==73]

In [ ]:

unpred_players = []
for player in players:
    # print('*****: ', player)
    if player not in pred_players:
        # print('------: ', player)
        unpred_players.append(player)

unpred_players_df = pd.DataFrame({'fpl_id':unpred_players,'value': 4,  'xPts': [0 for i in range(0, len(unpred_players))]})
player_with_pts = pd.concat([pred_pts, unpred_players_df])


player_with_pts

In [ ]:
player_with_pts = player_with_pts.merge(fpl_players[['id', 'price', 'name', 'position']], left_on='fpl_id', right_on='id', how='left')
layer_with_pts.drop(columns=['id'], inplace=True)
player_with_pts['element'] = player_with_pts['element'].astype(int)
player_with_pts

In [ ]:
player_with_pts = player_with_pts[player_with_pts['position']!= 5]
player_with_pts

In [ ]:
gks = player_with_pts[player_with_pts['position'] == 1]
defs = player_with_pts[player_with_pts['position'] == 2]
mids = player_with_pts[player_with_pts['position'] == 3]
fwds = player_with_pts[player_with_pts['position'] == 4]

gk_tup  = tuple([0 for i in range(len(gks))])
def_tup = tuple([0 for i in range(len(defs))])
mid_tup = tuple([0 for i in range(len(mids))])
fwd_tup = tuple([0 for i in range(len(fwds))])
len(fwd_tup), len(def_tup), len(mid_tup), len(gk_tup)
t = tuple([0 for i in range(len(gks))])
gks

In [ ]:
manager_1 = managers.iloc[780,:]

manager_1_picks = eval(manager_1['picks'])
list(manager_1_picks)

gk_1 = [player for player in manager_1_picks if player in gks['fpl_id'].values]
def_1 = [player for player in manager_1_picks if player in defs['fpl_id'].values]
mid_1 = [player for player in manager_1_picks if player in mids['fpl_id'].values]
fwd_1 = [player for player in manager_1_picks if player in fwds['fpl_id'].values]

manager_1_picks, gk_1, def_1, mid_1, fwd_1

## Model


In [ ]:

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from scipy.optimize import linprog
import pymc as pm

# Generate synthetic player data
np.random.seed(42)
n_players = 50  # Number of available players

In [ ]:
# Optimization: Selecting best team under budget constraint
budget = 100  # Total fantasy budget
C = 15  # Squad size
# c = data['cost'].values  # Player costs
# p = data['predicted_points_rf'].values  # Player points (Random Forest prediction)

A_eq = [np.ones(len(player_with_pts['xPts'].values))]  # Ensure exactly C players are selected
b_eq = [C]

bounds = [(0, 1) for _ in range(len(player_with_pts['xPts'].values))]  # Binary selection constraint

c = player_with_pts['price'].values  # Player costs
p = player_with_pts['xPts'].values  # Player points (Random Forest prediction)


# res = linprog(-p, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
# selected_players = data.iloc[np.where(res.x > 0.5)]

print("Selected Team (Scikit-Learn Optimization):")
# print(selected_players[['player_id', 'position', 'cost', 'predicted_points_rf']])


In [ ]:
pm.Multinomial('likelihood', n=1, p=2, value=c, observed=True)

In [ ]:
import numpy as np
import pymc as pm
np.random.seed(42)  # Use this before multiprocessing

# Bayesian Modeling with PyMC
with pm.Model() as model:
    alpha = np.ones(50)  # Prior for player selection probabilities
    player_selection = pm.Dirichlet('player_selection', a=alpha)
    observed_selection = pm.Multinomial('observed_selection', n=C, p=player_selection, observed=np.random.randint(0, 3, size=50))
    trace = pm.sample(1000, return_inferencdata=True)

print("Bayesian Inference Completed")

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
%config InlineBackend.figure_format = 'retina'

# Initialize random number generator
RANDOM_SEED = 8927
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-darkgrid")

In [ ]:
# True parameter values
alpha, sigma = 1, 1
beta = [1, 2.5]

# Size of dataset
size = 100

# Predictor variable
X1 = np.random.randn(size)
X2 = np.random.randn(size) * 0.2

# Simulate outcome variable
Y = alpha + beta[0] * X1 + beta[1] * X2 + rng.normal(size=size) * sigma

In [ ]:
fig, axes = plt.subplots(1, 2, sharex=True, figsize=(10, 4))
axes[0].scatter(X1, Y, alpha=0.6)
axes[1].scatter(X2, Y, alpha=0.6)
axes[0].set_ylabel("Y")
axes[0].set_xlabel("X1")
axes[1].set_xlabel("X2")

In [ ]:
import pymc as pm
print(f"Running on PyMC v{pm.__version__}")

In [ ]:
basic_model = pm.Model()

with basic_model:
    # Priors for unknown model parameters
    alpha = pm.Normal("alpha", mu=0, sigma=10)
    beta = pm.Normal("beta", mu=0, sigma=10, shape=2)
    sigma = pm.HalfNormal("sigma", sigma=1)

    # Expected value of outcome
    mu = alpha + beta[0] * X1 + beta[1] * X2

    # Likelihood (sampling distribution) of observations
    Y_obs = pm.Normal("Y_obs", mu=mu, sigma=sigma, observed=Y)"

In [ ]:

   # import pymc3 as pm
   import numpy as np

    def generate_Wo(f_params, X_params, c, Blb, q, O, W, Two):
        Wo = []

        # Step 1: Calculate beta values
        betas = [np.exp(X * f) for X, f in zip(X_params, f_params)]

        # Step 2: Generate Dirichlet distributions
        with pm.Model() as model:
            p_values = [pm.Dirichlet(f'p_{i}', a=beta) for i, beta in enumerate(betas)]

        # Step 3: Iterate over O
        for o in range(O):
            Stack = np.random.binomial(1, q)
            Reject = True

            while Reject:
                # Step 7: Generate multinomial samples
                with model:
                    k_values = [pm.Multinomial(f'k_{i}', n=1, p=p).random() for i, p in enumerate(p_values)]
                    k_values = [np.argmax(k) for k in k_values]

                if Stack == 1:
                    # Step 9: Replace km(1) with main m from team of kg
                    k_values[0] = k_values[0]  # Assuming main m is the first element

                # Step 11: Create portfolio wo
                wo = k_values

                # Step 12: Check conditions
                if wo in W and c * Two <= Blb:
                    Reject = False
                    Wo.append(wo)

        # Step 18: Return Wo
        return Wo

    # Example usage
    f_params = [0.1, 0.2, 0.3]
    X_params = [1.0, 2.0, 3.0]
    c = 0.5
    Blb = 10
    q = 0.7
    O = 5
    W = [[0, 1, 2], [1, 2, 0], [2, 0, 1]]  # Example set W
    Two = 1  # Example value for Two

    Wo = generate_Wo(f_params, X_params, c, Blb, q, O, W, Two)
    print(Wo)

## Current gwk picks


In [ ]:
import requests
base_url = 'https://fantasy.premierleague.com/api/' # leagues-h2h-matches/league/2125120
# https://fantasy.premierleague.com/api/leagues-h2h-matches/league/2520546/?page=1&event=28"


In [ ]:
league = 2520546
manager = 10118447
opp_manager = None
gwk_next = 28
gw_matches = requests.get(
                base_url + "leagues-h2h-matches/league/" + str(league) + "/?event=" + str(gwk_next)).json()
gw_picks = requests.get(
                base_url + "entry/" + str(manager) + "/event/" + str(gwk_next-1) + "/picks/").json()
picks = tuple([player['element']
                           for player in gw_picks['picks']])
print(picks)

[opp for opp in gw_matches['results'] if (opp['entry_1_entry'] == manager or opp['entry_2_entry'] == manager )]

for opp in gw_matches['results']:
    print(manager, opp)
    if (opp['entry_1_entry'] == manager):
        # Check if opponent is AVERAGE
        if opp['entry_2_name'] == 'AVERAGE':
            continue
        opp_manager = opp['entry_2_entry']
    elif (opp['entry_2_entry'] == manager):
        if opp['entry_1_name'] == 'AVERAGE':
            continue
        opp_manager = opp['entry_1_entry']


opp_gw_picks = requests.get(
                base_url + "entry/" + str(opp_manager) + "/event/" + str(gwk_next-1) + "/picks/").json()
opp_picks = tuple([player['element']
                           for player in opp_gw_picks['picks']])

In [ ]:
gks_all = gks['fpl_id'].tolist()
defs_all = defs['fpl_id'].tolist()
mids_all = mids['fpl_id'].tolist()
fwds_all = fwds['fpl_id'].tolist()

In [ ]:
k_1 = [player for player in opp_picks if player in gks['fpl_id'].values]
def_1 = [player for player in opp_picks if player in defs['fpl_id'].values]
mid_1 = [player for player in opp_picks if player in mids['fpl_id'].values]
fwd_1 = [player for player in opp_picks if player in fwds['fpl_id'].values]

In [ ]:
list(picks), gk_1, def_1, mid_1, fwd_1

In [ ]:
opp_gk_tuple = list(np.zeros(len(gks), dtype=int))
for player in gk_1:
    index = gks_all.index(player)
    opp_gk_tuple[index] = 1

## Player combinations


In [ ]:
from itertools import  combinations, product
formations = [[1,3,4,3], [1,3,5,2], [1,4,3,3], [1,4,4,2], [1,4,5,1], [1,5,2,3], [1,5,3,2], [1,5,4,1]]

In [ ]:
#==> No transfer made
# Create a dictionary to store combinations for each formation
forms_dict = {}
combs_dict = {}

for idx, formation in enumerate(formations):
    form_name = f"forms_{idx}"

    # for formation in formations:
    gk_combos = list(combinations(gk_1, formation[0]))
    def_combos = list(combinations(def_1, formation[1]))
    mid_combos = list(combinations(mid_1, formation[2]))
    fwd_combos = list(combinations(fwd_1, formation[3]))

    # Store the combinations in the dictionary
    forms_dict[form_name] = [gk_combos, def_combos, mid_combos, fwd_combos]

# Generate all possible combinations by taking one tuple from each group
for idx, combs in enumerate(forms_dict):
    comb_name = f"combinations_{idx+1}
    combinations_ = list(product(*forms_dict[combs]))

    combs_dict[comb_name] = combinations_

len(combs_dict['combinations_1'])+ len(combs_dict['combinations_2'])+ len(combs_dict['combinations_3'])+ len(combs_dict['combinations_4'])+ len(combs_dict['combinations_5']) + len(combs_dict['combinations_6'])+ len(combs_dict['combinations_7'])+ len(combs_dict['combinations_8'])


In [ ]:

# # One transfer is made
# gk_trans_1 = list(combinations(gks_all, 1))
# def_trans_1 =  list(combinations(defs_all, 1))
# mid_trans_1 =  list(combinations(mids_all, 1))
# fwd_trans_1 =  list(combinations(fwds_all, 1))



# # # Create a dictionary to store combinations for each formation

# gk_combos = list(combinations(gk_trans_1, 1))
# def_combos = list(combinations(def_trans_1, 3))
# mid_combos = list(combinations(mid_trans_1, 4))
# fwd_combos = list(combinations(fwd_trans_1, 3))
# trans_1 = len(gk_combos)*len(def_combos)* len(mid_combos)* len(fwd_combos)

# trans_1

In [ ]:
# # 2 transfers is made
# gk_trans_2 = list(combinations(gks_all, 2))
# def_trans_2 =  list(combinations(defs_all, 2))
# mid_trans_2 =  list(combinations(mids_all, 2))
# fwd_trans_2 =  list(combinations(fwds_all, 2))

# # # Create a dictionary to store combinations for each formation
# gk_combos = list(combinations(gk_trans_2, 1))
# def_combos = list(combinations(def_trans_2, 3))
# mid_combos = list(combinations(mid_trans_2, 4))
# fwd_combos = list(combinations(fwd_trans_2, 3))
# trans_2 = len(gk_combos)*len(def_combos)* len(mid_combos)* len(fwd_combos)

# trans_2

In [ ]:
import requests
import pandas as pd

# Step 1: Get the player's team picks for a specific gameweek (replace `team_id` & `gw`)
team_id = 123456  # Replace with actual team ID
gw = 25  # Replace with the gameweek number
url = f\"https://fantasy.premierleague.com/api/entry/{team_id}/event/{gw}/picks/\"

response = requests.get(url).json()
picks = pd.DataFrame(response['picks'])  # Extract player picks

# Step 2: Get the official FPL player data
players_url = \"https://fantasy.premierleague.com/api/bootstrap-static/\"
players_data = requests.get(players_url).json()
players_df = pd.DataFrame(players_data['elements'])  # All players
players_df['now_cost'] = players_df['now_cost'].copy()/10 # cost is in 10s

# Step 3: Merge to get the correct player names
merged_picks = picks.merge(players_df, left_on='element', right_on='id', how='left')

# Show the manager’s selected players with correct names
merged_picks[['element', 'web_name', 'team', 'multiplier', 'element_type_x', 'now_cost']]
players_df[[ 'web_name', 'team', 'now_cost']]

In [ ]:
pred = pd.read_csv('../FPL predictors/with new features/predicted.csv'),

pred_pts = pred[['xPts', 'value', 'fpl_id', 'element', 'value']].copy(),

pred_pts['fpl_id'] = pred_pts['fpl_id'].astype(int),

# divide the value by 10,
# pred_pts.loc[:, 'value'] = pred_pts['value'] / 10,
# pred_pts = pred_pts.apply(lambda x: x/10 for  x in players else None, axis=1).dropna(),
pred_players = pred_pts['fpl_id'].tolist(),
pred_pts[pred_pts['fpl_id'] ==443]

## Get player details


In [3]:
players_preds = pd.read_csv('./../FPL predictors/with new features/predicted.csv')
# print(players_preds)
players_preds = players_preds.dropna(axis=1).copy()
players_preds = players_preds.drop(['event', 'understat_id', 'xg_pred', 'xa_pred', 'cs_pred', 'yc_pred', 'sv_pred', 'h_team', 'a_team','was_home'], axis=1)
player_cols = players_preds.columns.tolist()

gk_preds = players_preds[players_preds['position'] == 'GK'][player_cols]
def_preds = players_preds[players_preds['position'] == 'DEF'][player_cols]
mid_preds = players_preds[players_preds['position'] == 'MID'][player_cols]
fwd_preds = players_preds[players_preds['position'] == 'FWD'][player_cols]


gk_preds = gk_preds.drop(['position'], axis=1)
def_preds = def_preds.drop(['position'], axis=1)
mid_preds = mid_preds.drop(['position'], axis=1)
fwd_preds = fwd_preds.drop(['position'], axis=1)

In [4]:
gw = 28
data_22_23 = pd.read_csv('../FPL predictors/with new features/data/joint/22-23/merged_player_data.csv').dropna()
data_23_24 = pd.read_csv('../FPL predictors/with new features/data/joint/23-24/merged_player_data.csv').dropna()
data_24_25 = pd.read_csv('../FPL predictors/with new features/data/joint/24-25/merged_player_data.csv')
data_tar = data_24_25[data_24_25['event']==gw]
data_24_25_ = data_24_25[data_24_25['event'] != gw]

data = pd.concat([data_22_23, data_23_24, data_24_25_])

data_cols = player_cols.copy()
data_cols.pop(data_cols.index('xPts'))
data_cols.pop(data_cols.index('fpl_id'))
data_cols.pop(data_cols.index('player_team'))

'player_team'

In [5]:
from sklearn.preprocessing import minmax_scale
# scaler = StandardScaler()
data_ = data[data_cols]
gk_data = data[data['position'] == 'GK'][data_cols]
def_data = data[data['position'] == 'DEF'][data_cols]
mid_data = data[data['position'] == 'MID'][data_cols]
fwd_data = data[data['position'] == 'FWD'][data_cols]

gk_data = gk_data.select_dtypes(include=[np.number])
gk_data = gk_data.drop(['expected_assists_3', 'expected_goal_involvements_3', 'expected_goals_3', 'expected_goals_3', 'goals_scored_3', 'own_goals_3', 'goals_3', 'shots_3', 'xG_3',
                        'npg_3', 'npxG_3', 'xGChain_3', 'xGBuildup_3', 'xA_3', 'assists_y_3', 'expected_goals_5', 'expected_assists_5', 'expected_goal_involvements_5', 'expected_goals_5', 'expected_goals_5', 'goals_scored_5', 'own_goals_5', 'goals_5', 'shots_5', 'xG_5',
                        'npg_5', 'npxG_5', 'xGChain_5', 'xGBuildup_5', 'xA_5', 'assists_y_5'], axis=1)
gk_data = pd.DataFrame(minmax_scale(gk_data), columns=gk_data.columns, index=gk_data.index)

def_data = def_data.select_dtypes(include=[np.number])
def_data = def_data.drop(['expected_assists_3', 'expected_goal_involvements_3', 'expected_goals_3', 'expected_goals_3', 'goals_scored_3', 'own_goals_3', 'goals_3', 'shots_3', 'xG_3',
                        'npg_3', 'npxG_3', 'xGChain_3', 'xGBuildup_3', 'xA_3', 'assists_y_3', 'expected_goals_5', 'expected_assists_5', 'expected_goal_involvements_5', 'expected_goals_5', 'expected_goals_5', 'goals_scored_5', 'own_goals_5', 'goals_5', 'shots_5', 'xG_5',
                        'npg_5', 'npxG_5', 'xGChain_5', 'xGBuildup_5', 'xA_5', 'assists_y_5','whh','whd', 'wha'], axis=1)
def_data = pd.DataFrame(minmax_scale(def_data), columns=def_data.columns, index=def_data.index)

mid_data = mid_data.select_dtypes(include=[np.number])
mid_data = mid_data.drop(['expected_assists_3', 'expected_goal_involvements_3', 'expected_goals_3', 'expected_goals_3', 'goals_scored_3', 'own_goals_3', 'goals_3', 'shots_3', 'xG_3',
                        'npg_3', 'npxG_3', 'xGChain_3', 'xGBuildup_3', 'xA_3', 'assists_y_3', 'expected_goals_5', 'expected_assists_5', 'expected_goal_involvements_5', 'expected_goals_5', 'expected_goals_5', 'goals_scored_5', 'own_goals_5', 'goals_5', 'shots_5', 'xG_5',
                        'npg_5', 'npxG_5', 'xGChain_5', 'xGBuildup_5', 'xA_5', 'assists_y_5','whh','whd', 'wha'], axis=1)
mid_data = pd.DataFrame(minmax_scale(mid_data), columns=mid_data.columns, index=mid_data.index)

fwd_data = fwd_data.select_dtypes(include=[np.number])
fwd_data = pd.DataFrame(minmax_scale(fwd_data), columns=fwd_data.columns, index=fwd_data.index)


## Wo Component


In [6]:
def get_beta(data):
    # Ensure data is numeric
    data = data.select_dtypes(include=[np.number])  # Drop non-numeric columns

    # Shape parameters
    np.random.seed(42)
    num_players = data.shape[0]
    num_features = data.shape[1]

    print(f"Number of Players: {num_players}, Features: {num_features}")

    # Convert to NumPy with float type
    X = data.to_numpy(dtype=np.float64)
    X = X.reshape((num_players, num_features))
    print(X.shape)

    # Observed probability distribution (Ensure valid probabilities)
    p = np.abs(np.random.randn(num_players, ))
    p /= p.sum( keepdims=True)  # Normalize to sum to 1
    print("p shape:", p.shape)

    with pm.Model() as model:
        # Prior: Uniform distribution for β
        beta = pm.Uniform("beta", lower=-1, upper=1, shape=(num_features,))
        print("Beta shape:", beta.shape)

        # Compute α = exp(X β) and ensure correct shape
        epsilon = 1e-3
        alpha = pm.math.exp(pm.math.dot(X, beta)) + epsilon
        print("Alpha shape:", alpha.shape)

        # Dirichlet likelihood (Ensure shape matches p)
        p = pm.Dirichlet("p", a=alpha, observed=p)
        # Debugging step to check the model
        model.debug(verbose=True)
        # MCMC Sampling
        trace = pm.sample(2000, return_inferencedata=True, target_accept=0.9)

    # Get estimated β values
    beta_estimated = trace.posterior["beta"].mean(dim=["chain", "draw"])
    print("Estimated β values:", beta_estimated.values)
    return beta_estimated

In [ ]:
beta_gk = get_beta(gk_data)

In [ ]:
beta_def = get_beta(def_data)

In [ ]:
beta_mid = get_beta(mid_data)

In [ ]:
beta_fwd = get_beta(fwd_data)

In [ ]:
len(def_beta)

In [7]:
from sklearn.preprocessing import minmax_scale
# scaler = StandardScaler()
data_ = data_tar[data_cols]
gk_data_tar = data_tar[data_tar['position'] == '1'][data_cols]
def_data_tar = data_tar[data_tar['position'] == '2'][data_cols]
mid_data_tar = data_tar[data_tar['position'] == '3'][data_cols]
fwd_data_tar = data_tar[data_tar['position'] == '4'][data_cols]


gk_data_tar = gk_data_tar.select_dtypes(include=[np.number])
gk_data_tar = gk_data_tar.drop(['expected_assists_3', 'expected_goal_involvements_3', 'expected_goals_3', 'expected_goals_3', 'goals_scored_3', 'own_goals_3', 'goals_3', 'shots_3', 'xG_3',
                        'npg_3', 'npxG_3', 'xGChain_3', 'xGBuildup_3', 'xA_3', 'assists_y_3', 'expected_goals_5', 'expected_assists_5', 'expected_goal_involvements_5', 'expected_goals_5', 'expected_goals_5', 'goals_scored_5', 'own_goals_5', 'goals_5', 'shots_5', 'xG_5',
                        'npg_5', 'npxG_5', 'xGChain_5', 'xGBuildup_5', 'xA_5', 'assists_y_5'], axis=1)
gk_data_tar = pd.DataFrame(minmax_scale(gk_data_tar), columns=gk_data_tar.columns, index=gk_data_tar.index)

def_data_tar = def_data_tar.select_dtypes(include=[np.number])
def_data_tar = def_data_tar.drop(['expected_assists_3', 'expected_goal_involvements_3', 'expected_goals_3', 'expected_goals_3', 'goals_scored_3', 'own_goals_3', 'goals_3', 'shots_3', 'xG_3',
                        'npg_3', 'npxG_3', 'xGChain_3', 'xGBuildup_3', 'xA_3', 'assists_y_3', 'expected_goals_5', 'expected_assists_5', 'expected_goal_involvements_5', 'expected_goals_5', 'expected_goals_5', 'goals_scored_5', 'own_goals_5', 'goals_5', 'shots_5', 'xG_5',
                        'npg_5', 'npxG_5', 'xGChain_5', 'xGBuildup_5', 'xA_5', 'assists_y_5','whh','whd', 'wha'], axis=1)
def_data_tar = pd.DataFrame(minmax_scale(def_data_tar), columns=def_data_tar.columns, index=def_data_tar.index)

mid_data_tar = mid_data_tar.select_dtypes(include=[np.number])
mid_data_tar = mid_data_tar.drop(['expected_assists_3', 'expected_goal_involvements_3', 'expected_goals_3', 'expected_goals_3', 'goals_scored_3', 'own_goals_3', 'goals_3', 'shots_3', 'xG_3',
                        'npg_3', 'npxG_3', 'xGChain_3', 'xGBuildup_3', 'xA_3', 'assists_y_3', 'expected_goals_5', 'expected_assists_5', 'expected_goal_involvements_5', 'expected_goals_5', 'expected_goals_5', 'goals_scored_5', 'own_goals_5', 'goals_5', 'shots_5', 'xG_5',
                        'npg_5', 'npxG_5', 'xGChain_5', 'xGBuildup_5', 'xA_5', 'assists_y_5','whh','whd', 'wha'], axis=1)
mid_data_tar = pd.DataFrame(minmax_scale(mid_data_tar), columns=mid_data_tar.columns, index=mid_data_tar.index)

fwd_data_tar = fwd_data_tar.select_dtypes(include=[np.number])
fwd_data_tar = pd.DataFrame(minmax_scale(fwd_data_tar), columns=fwd_data_tar.columns, index=fwd_data_tar.index)


In [8]:
beta_gk_ = pd.read_csv('./gk_beta.csv').values.squeeze()
beta_def_ = pd.read_csv('./def_beta.csv').values.squeeze()
beta_mid_ = pd.read_csv('./mid_beta.csv').values.squeeze()
beta_fwd_ = pd.read_csv('./fwd_beta.csv').values.squeeze()

alpha_gk_pred = np.exp(np.dot(gk_data_tar, beta_gk_.T))
alpha_def_pred = np.exp(np.dot(def_data_tar, beta_def_.T))
alpha_mid_pred = np.exp(np.dot(mid_data_tar, beta_mid_.T))
alpha_fwd_pred = np.exp(np.dot(fwd_data_tar, beta_fwd_.T))

In [9]:
dir_gk = np.random.dirichlet(alpha_gk_pred).tolist()
dir_def = np.random.dirichlet(alpha_def_pred).tolist()
dir_mid = np.random.dirichlet(alpha_mid_pred).tolist()
dir_fwd = np.random.dirichlet(alpha_fwd_pred).tolist()

In [10]:
dir_ = tuple([dir_gk, dir_def, dir_mid, dir_fwd])

In [ ]:
f = len(beta_gk_)  # Number of group

# Step 2: Define and sample from Dirichlet distributio
with pm.Model() as model:
    p_gk = pm.Dirichlet(p_gk, a=alpha_gk_pred)  # Ensure correct shap
    p_def = pm.Dirichlet(p_def, a=alpha_def_pred)  # Ensure correct shap
    p_mid = pm.Dirichlet(p_mid, a=alpha_mid_pred)  # Ensure correct shap
    p_fwd = pm.Dirichlet(p_fwd, a=alpha_fwd_pred)  # Ensure correct shap

    trace = pm.sample_prior_predictive(samples=1)

SyntaxError: expected ':' (4128212464.py, line 4)

In [11]:
# Extract samples correctly
sampled_p_gk = trace.prior["p_gk"].values.squeeze()  # Ensure shape is (f, num_features)
sampled_p_def = trace.prior["p_def"].values.squeeze()  # Ensure shape is (f, num_features)
sampled_p_mid = trace.prior["p_mid"].values.squeeze()  # Ensure shape is (f, num_features)
sampled_p_fwd = trace.prior["p_fwd"].values.squeeze()  # Ensure shape is (f, num_features)"


NameError: name 'trace' is not defined

In [ ]:
W = set()  # Placeholder for valid portfolios
Wo = []  # List to store accepted portfolios

In [ ]:
gk = np.random.multinomial(1, sampled_p_gk)
def_ = np.random.multinomial(3, sampled_p_def)
mid = np.random.multinomial(4, sampled_p_mid)
fwd = np.random.multinomial(4, sampled_p_fwd)
gk, def_, mid, fwd

gk = np.random.multinomial(1, sampled_p_gk)
gk_idx = [i for i, x in enumerate(gk) if x == 1]
def_idx = [i for i, x in enumerate(def_) if x == 1]
mid_idx = [i for i, x in enumerate(mid) if x == 1]

#  If stacking

fwd_idx = [i for i, x in enumerate(fwd) if x == 1]

gk_idx, def_idx, mid_idx, fwd_idx

In [ ]:
import numpy as np
import pymc as pm
import scipy.stats as stats

# === Step 1: Define Constants ===
P = 100  # Total available players
C = 15  # Squad size
cin = 11  # Starting lineup size
budget = 100  # Maximum budget
Blb = 95  # Lower bound on spent budget
O = 1000  # Number of squad simulations

# Example positions: Assume these are valid players for selection
Pg, Pd, Pm, Pf = len(gk_data_tar), len(def_data_tar), len(mid_data_tar), len(fwd_data_tar)  # Players per position
positions = {"g": Pg, "d": Pd, "m": Pm, "f": Pf}

# === Step 2: Generate Dirichlet Parameters for Position Selection ===
with pm.Model():
    alpha_g = np.exp(np.dot(gk_data_tar, beta_gk_.T))
    alpha_d = np.exp(np.dot(def_data_tar, beta_def_.T))
    alpha_m = np.exp(np.dot(mid_data_tar, beta_mid_.T))
    alpha_f = np.exp(np.dot(fwd_data_tar, beta_fwd_.T))

    pg = pm.Dirichlet("pg", a=alpha_g)  # Ensure correct shape
    pd = pm.Dirichlet("pd", a=alpha_d)  # Ensure correct shape
    pmid = pm.Dirichlet("pm", a=alpha_m)  # Ensure correct shape
    pf = pm.Dirichlet("pf", a=alpha_f)  # Ensure correct shape

    trace = pm.sample(2000, tune=1000, chains=4, target_accept=50 , return_inferencedata=True)

# Extract posterior means
pg, pd, pmid, pf = (
    trace.posterior["pg"].mean(dim=("chain", "draw")).values,
    trace.posterior["pd"].mean(dim=("chain", "draw")).values,
    trace.posterior["pm"].mean(dim=("chain", "draw")).values,
    trace.posterior["pf"].mean(dim=("chain", "draw")).values,)

In [ ]:
import arviz as az
az.plot_trace(data=trace, var_names=["pg", "pd", 'pm', 'pf'])

In [ ]:
import pandas as pd__
pg_ = pd__.DataFrame({'gk': pg})
pd_ = pd__.DataFrame({'def': pd})
pmid_ = pd__.DataFrame({'mid': pmid})
pf_ = pd__.DataFrame({'fwd': pf})

pg_.to_csv('./gk_beta_post.csv', index=False)
pd_.to_csv('./def_beta_post.csv', index=False)
pmid_.to_csv('./mid_beta_post.csv', index=False)
pf_.to_csv('./fwd_beta_post.csv', index=False)

In [ ]:
# === Step 1: Define Constants ===
P = 100  # Total available players
C = 15  # Squad size
cin = 11  # Starting lineup size
budget = 100  # Maximum budget
Blb = 95  # Lower bound on spent budget
O = 500_000  # Number of squad simulations
q = 0.25  # Assume 30% stacking probability

Wo = set()
# === Step 3: Generate Possible Team Selections (Algorithm 1) ===
def generate_squad(pg, pd, pmid, pf, q, budget):

    """Generate a valid team selection for an opponent."""
    stack = np.random.binomial(1, q)  # Stacking decision

    while True:
        selected_g = np.random.multinomial(2, pg)
        if set(selected_g) <= {0, 1}:  # Ensure only 1s and 0s exist
            break

    while True:
        selected_d = np.random.multinomial(5, pd)
        if set(selected_d) <= {0, 1}:  # Ensure only 1s and 0s exist
            break

    while True:
        selected_m = np.random.multinomial(5, pmid)
        if set(selected_m) <= {0, 1}:  # Ensure only 1s and 0s exist
            break
    while True:
        selected_f = np.random.multinomial(3, pf)
        if set(selected_f) <= {0, 1}:  # Ensure only 1s and 0s exist
            break

    # selected_g = np.random.multinomial(2, pg)
    # sel_g_ = [idx for idx, _ in enumerate(selected_g)]

    # selected_d = np.random.multinomial(5, pd)
    # selected_m = np.random.multinomial(5, pmid)
    # selected_f = np.random.multinomial(3, pf)

    # Apply stacking (if enabled, pick an extra midfielder from the same club as a forward)
    if stack:
        selected_m[0] = selected_f[0]  # Stack a midfielder with a forward

    g_cost = np.dot(selected_g, g_prices)[0]
    d_cost = np.dot(selected_d, d_prices)[0]
    m_cost = np.dot(selected_m, m_prices)[0]
    f_cost = np.dot(selected_f, f_prices)[0]

    # Ensure total budget is within limits
    squad = np.concatenate([selected_g, selected_d, selected_m, selected_f])

    # Check  if no players no more than 3 players from the same team are selected
    # print(data_tar['player_team'])
    total_cost = sum([g_cost, d_cost, m_cost, f_cost])
    # print(total_cost)
    # print([g_cost, d_cost, m_cost, f_cost])
    if total_cost > budget or total_cost > Blb:
    #     print('Above =============================================================>')
        return None  # Reject and retry
    return squad

# Generate O possible squads
while len(Wo)  < O:
    squad = generate_squad(pg_curr, pd_curr, pm_curr, pf_curr, q, budget)
    # print('*****: ', squad)
    if squad is not None:
        # print('=========================================>', squad)

        Wo.add(tuple(squad))

    # print(Wo)
    print(f'-------------------------------> {len(Wo)} ===> {(len(Wo)/O)*100}%')
# Print example squad
print("Example Generated Squad:", len(Wo))

## Wo


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import minmax_scale

In [3]:
players_preds = pd.read_csv('./../FPL predictors/with new features/predicted.csv')
# print(players_preds)
players_preds = players_preds.dropna(axis=1).copy()
players_preds = players_preds.drop(['event', 'understat_id', 'xg_pred', 'xa_pred', 'cs_pred', 'yc_pred', 'sv_pred', 'h_team', 'a_team','was_home'], axis=1)
player_cols = players_preds.columns.tolist()


In [4]:
gw = 28
data_22_23 = pd.read_csv('../FPL predictors/with new features/data/joint/22-23/merged_player_data.csv').dropna()
data_23_24 = pd.read_csv('../FPL predictors/with new features/data/joint/23-24/merged_player_data.csv').dropna()
data_24_25 = pd.read_csv('../FPL predictors/with new features/data/joint/24-25/merged_player_data.csv')
data_tar = data_24_25[data_24_25['event']==gw]
data_24_25_ = data_24_25[data_24_25['event'] != gw]

data = pd.concat([data_22_23, data_23_24, data_24_25_])

data_cols = player_cols.copy()
data_cols.pop(data_cols.index('xPts'))
data_cols.pop(data_cols.index('fpl_id'))
data_cols.pop(data_cols.index('player_team'))

'player_team'

In [5]:
data_22_23 = pd.read_csv('../FPL predictors/with new features/data/joint/22-23/merged_player_data.csv').dropna()
data_23_24 = pd.read_csv('../FPL predictors/with new features/data/joint/23-24/merged_player_data.csv').dropna()
data_24_25 = pd.read_csv('../FPL predictors/with new features/data/joint/24-25/merged_player_data.csv')
data_tar = data_24_25[data_24_25['event']==gw]
data_24_25_ = data_24_25[data_24_25['event'] != gw]

data_24_25_ = data_24_25[data_24_25['event'] != 28]

data = pd.concat([data_22_23, data_23_24, data_24_25_])


In [6]:

# scaler = StandardScaler()
data_ = data_tar[data_cols]
gk_data_tar = data_tar[data_tar['position'] == '1'][data_cols]
def_data_tar = data_tar[data_tar['position'] == '2'][data_cols]
mid_data_tar = data_tar[data_tar['position'] == '3'][data_cols]
fwd_data_tar = data_tar[data_tar['position'] == '4'][data_cols]


gk_data_tar = gk_data_tar.select_dtypes(include=[np.number])
gk_data_tar = gk_data_tar.drop(['expected_assists_3', 'expected_goal_involvements_3', 'expected_goals_3', 'expected_goals_3', 'goals_scored_3', 'own_goals_3', 'goals_3', 'shots_3', 'xG_3',
                        'npg_3', 'npxG_3', 'xGChain_3', 'xGBuildup_3', 'xA_3', 'assists_y_3', 'expected_goals_5', 'expected_assists_5', 'expected_goal_involvements_5', 'expected_goals_5', 'expected_goals_5', 'goals_scored_5', 'own_goals_5', 'goals_5', 'shots_5', 'xG_5',
                        'npg_5', 'npxG_5', 'xGChain_5', 'xGBuildup_5', 'xA_5', 'assists_y_5'], axis=1)
gk_data_tar = pd.DataFrame(minmax_scale(gk_data_tar), columns=gk_data_tar.columns, index=gk_data_tar.index)

def_data_tar = def_data_tar.select_dtypes(include=[np.number])
def_data_tar = def_data_tar.drop(['expected_assists_3', 'expected_goal_involvements_3', 'expected_goals_3', 'expected_goals_3', 'goals_scored_3', 'own_goals_3', 'goals_3', 'shots_3', 'xG_3',
                        'npg_3', 'npxG_3', 'xGChain_3', 'xGBuildup_3', 'xA_3', 'assists_y_3', 'expected_goals_5', 'expected_assists_5', 'expected_goal_involvements_5', 'expected_goals_5', 'expected_goals_5', 'goals_scored_5', 'own_goals_5', 'goals_5', 'shots_5', 'xG_5',
                        'npg_5', 'npxG_5', 'xGChain_5', 'xGBuildup_5', 'xA_5', 'assists_y_5','whh','whd', 'wha'], axis=1)
def_data_tar = pd.DataFrame(minmax_scale(def_data_tar), columns=def_data_tar.columns, index=def_data_tar.index)

mid_data_tar = mid_data_tar.select_dtypes(include=[np.number])
mid_data_tar = mid_data_tar.drop(['expected_assists_3', 'expected_goal_involvements_3', 'expected_goals_3', 'expected_goals_3', 'goals_scored_3', 'own_goals_3', 'goals_3', 'shots_3', 'xG_3',
                        'npg_3', 'npxG_3', 'xGChain_3', 'xGBuildup_3', 'xA_3', 'assists_y_3', 'expected_goals_5', 'expected_assists_5', 'expected_goal_involvements_5', 'expected_goals_5', 'expected_goals_5', 'goals_scored_5', 'own_goals_5', 'goals_5', 'shots_5', 'xG_5',
                        'npg_5', 'npxG_5', 'xGChain_5', 'xGBuildup_5', 'xA_5', 'assists_y_5','whh','whd', 'wha'], axis=1)
mid_data_tar = pd.DataFrame(minmax_scale(mid_data_tar), columns=mid_data_tar.columns, index=mid_data_tar.index)

fwd_data_tar = fwd_data_tar.select_dtypes(include=[np.number])
fwd_data_tar = pd.DataFrame(minmax_scale(fwd_data_tar), columns=fwd_data_tar.columns, index=fwd_data_tar.index)

In [7]:
pg_curr = pd.read_csv('./gk_beta_post.csv').values.squeeze()
pd_curr = pd.read_csv('./def_beta_post.csv').values.squeeze()
pm_curr = pd.read_csv('./mid_beta_post.csv').values.squeeze()
pf_curr = pd.read_csv('./fwd_beta_post.csv').values.squeeze()

g_prices = players_preds[players_preds['position'] == 'GK'][['xPts']]
d_prices = players_preds[players_preds['position'] == 'DEF'][['xPts']]
m_prices = players_preds[players_preds['position'] == 'MID'][['xPts']]
f_prices = players_preds[players_preds['position'] == 'FWD'][['xPts']]

g_xPts = players_preds[players_preds['position'] == 'GK']['xPts'].tolist()
d_xPts = players_preds[players_preds['position'] == 'DEF']['xPts'].tolist()
m_xPts = players_preds[players_preds['position'] == 'MID']['xPts'].tolist()
f_xPts = players_preds[players_preds['position'] == 'FWD']['xPts'].tolist()

g_teams = players_preds[players_preds['position'] == 'GK']['player_team'].tolist()
d_teams = players_preds[players_preds['position'] == 'DEF']['player_team'].tolist()
m_teams = players_preds[players_preds['position'] == 'MID']['player_team'].tolist()
f_teams = players_preds[players_preds['position'] == 'FWD']['player_team'].tolist()

g_ids = players_preds[players_preds['position'] == 'GK']['fpl_id'].tolist()
d_ids = players_preds[players_preds['position'] == 'DEF']['fpl_id'].tolist()
m_ids = players_preds[players_preds['position'] == 'MID']['fpl_id'].tolist()
f_ids = players_preds[players_preds['position'] == 'FWD']['fpl_id'].tolist()


all_xPts = g_xPts + d_xPts + m_xPts + f_xPts

In [8]:
from collections import Counter

# === Step 1: Define Constants ===
P = 100  # Total available players
C = 11  # Squad size
cin = 11  # Starting lineup size
budget = 100  # Maximum budget
Blb = 95  # Lower bound on spent budget"

O = 50  # Number of squad simulations

O = 5_000  # Number of squad simulations"

q = 0.25  # Assume 30% stacking probability

gk_sizes = [1]
def_sizes = [3,4,5]
mid_sizes = [2,3,4,5]
fwd_sizes = [1,2,3]
Wo = set()
# === Step 3: Generate Possible Team Selections (Algorithm 1) ===
def generate_squad(pg, pdef, pmid, pf, q, budget):

    """Generate a valid team selection for an opponent."""
    stack = np.random.binomial(1, q)  # Stacking decision


    # print('===============================> Adding Goalkeeper')

    while True:
        selected_g = np.random.multinomial(1, pg)
        if set(selected_g) <= {0, 1}:  # Ensure only 1s and 0s exist
            break
    g_cost = np.dot(selected_g, g_prices)[0]
    g_team_idx = [idx for idx, _ in enumerate(selected_g) if selected_g[idx] == 1]
    selected_team_g = [g_teams[i] for i in g_team_idx] # Get the selected teams
    squad_g = [g_ids[i] for i in g_team_idx]

    for i in def_sizes:
        # print('===============================> Adding Defender', i)
        while True: # substitute for a do_while loop
            while True:# Ensure only 1s and 0s exist
                selected_d = np.random.multinomial(i, pdef)
                if set(selected_d) <= {0, 1}:  # Ensure only 1s and 0s exist
                    break
            d_cost = np.dot(selected_d, d_prices)[0]
            d_team_idx = [idx for idx, _ in enumerate(selected_d) if selected_d[idx] == 1]
            selected_team_g_d = selected_team_g + [d_teams[i] for i in d_team_idx] # Get the selected teams
            squad_g_d = squad_g + [d_ids[i] for i in d_team_idx]

            if ((not any(team >=3 for team in Counter(selected_team_g_d).values())) and g_cost + d_cost < Blb):
                # Check if no more than 3 players from the same team are selected
                # If the cost is below the lower bound, break and continue to the next iteration
                break

        for j in mid_sizes:
            # print('===============================> Adding Midfielder ',i,  j)
            while True: # substitute for a do_while loop
                while True:
                    selected_m = np.random.multinomial(j, pmid)
                    if set(selected_m) <= {0, 1}:  # Ensure only 1s anm0s exist
                        break
                m_cost = np.dot(selected_m, m_prices)[0]
                m_team_idx = [idx for idx, _ in enumerate(selected_m) if selected_m[idx] == 1]
                selected_team_g_d_m = selected_team_g_d + [m_teams[i] for i in m_team_idx] # Get the selected teams
                squad_g_d_m = squad_g_d + [m_ids[i] for i in m_team_idx]

                if  ((not any(team >=3 for team in Counter(selected_team_g_d_m).values())) and g_cost + d_cost + m_cost < Blb):
                    # Check if no more than 3 players from the same team are selected
                    # If the cost is below the lower bound, break and continue to the next iteration
                    break

            for k in fwd_sizes:
                # print('===============================> Adding Forward ',i,  j, k)
                if(i+j+k < 10 or i+j+k > 10):
                    continue

                while True:
                    while True:
                        selected_f = np.random.multinomial(k, pf)
                        if set(selected_f) <= {0, 1}:  # Ensure only 1s anm0s exist
                            break
                    f_cost = np.dot(selected_f, f_prices)[0]

                    f_team_idx = [idx for idx, _ in enumerate(selected_f) if selected_f[idx] == 1]
                    selected_team_g_d_m_f = selected_team_g_d_m + [f_teams[i] for i in f_team_idx] # Get the selected teams
                    squad_g_d_m_f = squad_g_d_m + [f_ids[i] for i in f_team_idx]

                    squad_cost = g_cost + d_cost + m_cost + f_cost
                    print('_______________________: ',squad_cost)
                    if  ((not any(team >=3 for team in Counter(selected_team_g_d_m_f).values())) and  squad_cost < Blb): # 70 <

                    #if((not any(team >=3 for team in Counter(selected_team_g_d_m_f).values())) and g_cost + d_cost + m_cost + f_cost < Blb):

                        # Check if no more than 3 players from the same team are selected
                        # If the cost is below the lower bound, break and continue to the next iteration
                        Wo.add(tuple(squad_g_d_m_f))
                        break

squad = generate_squad(pg_curr, pd_curr, pm_curr, pf_curr, q, budget)
len(Wo)
# Generate O possible squads
while len(Wo)  < O:
    generate_squad(pg_curr, pd_curr, pm_curr, pf_curr, q, budget)
    print(f'-------------------------------> {len(Wo)} ===> {(len(Wo)/O)*100}%')
# Print example squad
print("Example Generated Squad:", len(Wo))

_______________________:  22.968763124636617
_______________________:  22.237103648567818
_______________________:  24.188881413133373
_______________________:  23.31177488457326
_______________________:  21.46182120442984
_______________________:  23.406012921853588
_______________________:  22.80639281795888
_______________________:  20.54195690918258
_______________________:  26.286386729325674
_______________________:  24.468969705281417
_______________________:  25.16048681695332
_______________________:  25.409464642697127
_______________________:  24.134514683169385
_______________________:  23.31608663073085
_______________________:  23.794720456825655
_______________________:  21.72705129461316
_______________________:  21.46969585283152
_______________________:  22.22128683172054
_______________________:  20.174575532862786
-------------------------------> 16 ===> 0.32%
_______________________:  23.190931704344145
_______________________:  24.3224380283607
___________________

In [9]:
c = 0
for id_ in list((51.0, 244.0, 50.0, 96.0, 182.0, 489.0, 701.0, 99.0, 123.0, 351.0, 421.0)):
    print(id_)
    c += players_preds[players_preds['fpl_id'] == id_]['value'].values[0]*10


print(c )

51.0
244.0
50.0
96.0
182.0
489.0
701.0
99.0
123.0
351.0
421.0
71.80000000000001


In [10]:
G_ = []
for wo in Wo:
    G_.append(np.sum([players_preds[players_preds['fpl_id'] == id_]['xPts'].values[0] for id_ in wo]))


In [11]:
Wo_ = []
player_ids  = players_preds['fpl_id'].tolist()
for i in Wo:
    # Create a tuple of 0s and 1s for each player
    squad = tuple([1 if id_ in i else 0 for id_ in player_ids])
    Wo_.append(squad)


In [12]:
available_ids = players_preds['fpl_id'].unique().tolist()
len(available_ids)

517

In [13]:
filtered_data = data[data['element'].isin(available_ids)]
filtered_data['element'].max()

772.0

In [14]:
# Group by 'event', sort the events in ascending order, and pivot the data
pivoted_data = pivoted_data = filtered_data.pivot_table(index='event', columns='element', values='xP')

# Reset the index to make 'event' a column again
# pivoted_data = pivoted_data.reset_index(drop=True)
# pivoted_data

In [15]:
pivoted_data.fillna(0, inplace=True)
# pivoted_data

In [16]:
pivoted_ids = pivoted_data.columns.tolist()
len(pivoted_ids)
extra_ids = list(set(available_ids) - set(pivoted_ids)) # Check for missing ids

In [17]:
# Add extra_ids as new columns with 0 values for each event
for element in extra_ids:
    pivoted_data[element] = 0

pivoted_data= pivoted_data.reindex(sorted(pivoted_data.columns), axis=1)

In [18]:
pivoted_data_with_extras = pivoted_data.reindex(sorted(pivoted_data.columns), axis=1)
# pivoted_data_with_extras

## Covariance Matrix


In [38]:
cov_mat = pivoted_data_with_extras.cov()
# cov_mat

In [39]:
cov_mat

element,2.0,4.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,...,731.0,732.0,734.0,755.0,757.0,762.0,764.0,765.0,770.0,772.0
element,,,,,,,,,,,,,,,,,,,,,
2.0,5.808231,-0.613140,0.700135,6.004884,1.585156,-0.260448,1.633224,2.034475,3.124908,1.460145,...,0.314168,0.020533,-0.022607,0.0,0.374840,0.403819,0.0,0.021230,0.0,0.0
4.0,-0.613140,0.823933,-0.403901,-0.717749,-0.491091,0.050740,-0.447854,-0.237309,-0.601550,-0.040284,...,-0.130261,-0.010057,-0.028016,0.0,-0.198983,-0.058905,0.0,-0.007183,0.0,0.0
6.0,0.700135,-0.403901,4.507836,2.221536,1.182586,1.949213,2.579352,-0.563712,1.173129,1.355530,...,1.054186,0.087677,0.228325,0.0,-0.086398,0.470311,0.0,0.104422,0.0,0.0
7.0,6.004884,-0.717749,2.221536,13.163309,3.319175,0.493001,3.622137,3.410725,3.411188,1.679956,...,0.741366,-0.009139,0.027166,0.0,0.254769,0.332183,0.0,-0.018883,0.0,0.0
8.0,1.585156,-0.491091,1.182586,3.319175,2.958895,0.659777,1.299531,0.714069,1.862745,0.281398,...,0.026015,-0.027440,0.039893,0.0,-0.605363,-0.109559,0.0,-0.017283,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
762.0,0.403819,-0.058905,0.470311,0.332183,-0.109559,0.212091,0.847506,0.208141,0.616828,0.733409,...,0.348298,0.121565,0.042119,0.0,-0.161550,0.887312,0.0,0.123898,0.0,0.0
764.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0
765.0,0.021230,-0.007183,0.104422,-0.018883,-0.017283,0.042674,0.111759,0.008013,0.119488,0.053580,...,0.066382,0.026031,0.010740,0.0,-0.019701,0.123898,0.0,0.026316,0.0,0.0


In [40]:
mu_delta = pivoted_data_with_extras.mean()
sigma_delta = pivoted_data_with_extras.std()

# Replace zeros with a small positive value. Ensure all standard deviations are positive
sigma_delta = np.where(sigma_delta <= 0, 1e-6, sigma_delta)

In [41]:
mu_delta.min()

-0.07894736842105263

In [42]:
v = np.exp(mu_delta)
v.min()

0.9240885594051769

In [43]:
sigma_delta.min()

1e-06

In [44]:
def monte_carlo_simulation(W_o, mu_delta, sigma_delta, num_samples=500):
    """Generate Monte Carlo samples for player performances and opponent scores using PyMC."""
    with pm.Model() as model:
        # Note that we access the distribution for the standard
        # deviations, and do not create a new random variable.
        sd_dist = pm.HalfNormal.dist(sigma_delta)

        mu_delta_ = pm.Deterministic('mu_delta_', pm.math.exp(mu_delta)+1e-6)
        chol, corr, sigmas = pm.LKJCholeskyCov(
            'chol_cov', eta=20, n=517, sd_dist=sd_dist
        )

        # if you only want the packed Cholesky:
        # packed_chol = pm.LKJCholeskyCov(
        #     'chol_cov', eta=4, n=10, sd_dist=sd_dist, compute_corr=False
        # )
        # chol = pm.expand_packed_triangular(10, packed_chol, lower=True)

        # Define a new MvNormal with the given covariance
        # vals = pm.MvNormal('vals', mu=np.zeros(517), chol=chol, shape=517)

        # # Or transform an uncorrelated normal:
        # vals_raw = pm.HalfNormal('vals_raw',sigma=1, shape=517)
        # vals = pt.dot(chol, vals_raw)

        # Or compute the covariance matrix
        cov = pt.dot(chol, chol.T)

        print(mu_delta_.min())
        delta = pm.MvNormal("delta", mu=mu_delta_, cov=cov, shape=517)

        print(delta)

        k = int(len(W_o) * 0.5)  # Index for 50th percentile

        G_r = pm.Deterministic("G_r", pt.sort(pt.dot(W_o, delta))[-int(len(W_o) * 0.5)])  #pt.sort(pt.dot(W_o, delta), k)[k])
        print(G_r)
        trace = pm.sample(num_samples, return_inferencedata=True, cores=2, progressbar=True)

    delta_samples = trace.posterior["delta"].values.reshape(-1, 517)
    G_r_samples = trace.posterior["G_r"].values.flatten()

    return delta_samples, G_r_samples

In [45]:
with pm.Model() as modelx:
    # Note that we access the distribution for the standard
    # deviations, and do not create a new random variable.
    sd_dist = pm.DiracDelta.dist( sigma_delta)
    chol, corr, sigmas = pm.LKJCholeskyCov(
        'chol_cov', eta=4, n=517, sd_dist=sd_dist
    )

    # if you only want the packed Cholesky:
    # packed_chol = pm.LKJCholeskyCov(
    #     'chol_cov', eta=4, n=10, sd_dist=sd_dist, compute_corr=False
    # )
    # chol = pm.expand_packed_triangular(10, packed_chol, lower=True)

    # Define a new MvNormal with the given covariance
    vals = pm.MvNormal('vals', mu=np.zeros(517), chol=chol, shape=517)

    # Or transform an uncorrelated normal:
    vals_raw = pm.Normal('vals_raw', mu=0, sigma=1, shape=517)
    vals = pt.dot(chol, vals_raw)

    # Or compute the covariance matrix
    cov = pt.dot(chol, chol.T)

In [46]:
samples_ = monte_carlo_simulation(Wo_, mu_delta, sigma_delta, num_samples=500)

Neg.0
delta
G_r


Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [chol_cov, delta]


Output()

Sampling 2 chains for 1_000 tune and 500 draw iterations (2_000 + 1_000 draws total) took 475 seconds.
c:\Users\Ilyas\anaconda3\envs\Pycaret\Lib\site-packages\arviz\stats\diagnostics.py:592: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
There were 489 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


In [47]:
delta_samples, G_r_samples = samples_

In [48]:
G_r_samples

array([73.55556613, 73.55556613, 73.55556613, 73.55556613, 73.55556613,
       73.55556613, 73.55556613, 73.55556613, 73.55556613, 73.55556613,
       73.55556613, 73.55556613, 73.55556613, 73.55556613, 73.55556613,
       73.55556613, 73.55556613, 73.55556613, 73.55556613, 73.55556613,
       73.55556613, 73.55556613, 73.55556613, 73.55556613, 73.55556613,
       73.55556613, 73.55556613, 73.55556613, 73.55556613, 73.55556613,
       73.55556613, 73.55556613, 73.55556613, 73.55556613, 73.55556613,
       73.55556613, 73.55556613, 73.55556613, 73.55556613, 73.55556613,
       73.55556613, 73.55556613, 73.55556613, 73.55556613, 73.55556613,
       73.55556613, 73.55556613, 73.55556613, 73.55556613, 73.55556613,
       73.55556613, 73.55556613, 73.55556613, 73.55556613, 73.55556613,
       73.55556613, 73.55556613, 73.55556613, 73.55556613, 73.55556613,
       73.55556613, 73.55556613, 73.55556613, 73.55556613, 73.55556613,
       73.55556613, 73.55556613, 73.55556613, 73.55556613, 73.55

In [49]:
import numpy as np

def estimate_parameters(delta_samples, G_r_samples):
    """
    Estimates parameters from Monte Carlo samples.

    Parameters:
    - delta_samples: Array of shape (num_samples, num_parameters) containing samples of delta.
    - G_r_samples: Array of shape (num_samples,) containing samples of G(r').

    Returns:
    - mu_delta: Mean of delta samples.
    - Sigma_delta: Covariance matrix of delta samples.
    - mu_G_r: Mean of G(r') samples.
    - sigma2_G_r: Variance of G(r') samples.
    - sigma_delta: Standard deviations of delta samples.
    - G_r: G(r') samples.
    """

    # 1. Estimate mu_delta (Mean of delta)
    mu_delta = np.mean(delta_samples, axis=0)

    # 2. Estimate Sigma_delta (Covariance matrix of delta)
    Sigma_delta = np.cov(delta_samples, rowvar=False)

    # 3. Estimate mu_G_r (Mean of G(r'))
    mu_G_r = np.mean(G_r_samples)

    # 4. Estimate sigma2_G_r (Variance of G(r'))
    sigma2_G_r = np.var(G_r_samples, ddof=1)  # Using unbiased estimator (Bessel's correction)

    # 5. Estimate sigma_delta (Standard deviations of delta)
    sigma_delta = np.std(delta_samples, axis=0, ddof=1)  # Using unbiased estimator

    # 6. G(r') samples
    G_r = G_r_samples

    return {
        'mu_delta': mu_delta,
        'Sigma_delta': Sigma_delta,
        'mu_G_r': mu_G_r,
        'sigma2_G_r': sigma2_G_r,
        'sigma_delta': sigma_delta,
        'G_r': G_r
    }

# Example usage:
parameters = estimate_parameters(delta_samples, G_r_samples)
print("Estimated Parameters:")
print("------------------------")
print(f"μδ: {parameters['mu_delta']}")
print(f"Σδ:\n{parameters['Sigma_delta']}")
print(f"μG(r′): {parameters['mu_G_r']}")
print(f"σ²G(r′): {parameters['sigma2_G_r']}")
print(f"σδ: {parameters['sigma_delta']}")
print(f"G(r′) samples: {parameters['G_r']}")


Estimated Parameters:
------------------------
μδ: [1.84761672e+01 1.63778894e+00 3.74149638e+01 1.21859828e+02
 6.45310904e+00 4.30532339e+00 2.76134388e+01 4.56957533e+00
 1.72576671e+01 7.55547034e+01 4.62144098e+01 1.67616912e+01
 1.44110914e+01 5.40094825e-01 1.35635973e+00 1.89944525e+02
 8.06752896e+01 2.64462440e+00 9.38512513e-01 1.51363373e+00
 1.58437067e+00 5.19399421e+00 4.75208872e+01 1.61864891e+00
 1.69742639e+01 6.44824607e+01 2.45136111e+01 3.33487015e+00
 1.15831624e+00 4.92901888e+00 2.59797745e+00 1.50986107e+01
 9.09264218e+00 1.11351847e+01 5.01060526e+00 2.17434409e+01
 1.06650197e+01 9.07567885e+00 1.23959587e+01 7.01668278e-01
 2.20459615e+00 3.89074307e+00 9.07660616e+00 7.29543662e+00
 2.18570309e+01 6.78430141e-01 1.51623421e+02 2.44415213e+00
 3.39573515e+00 4.50622275e+00 6.07911118e+00 7.73296589e-01
 2.53786248e+00 8.19880075e+00 1.91532000e+00 1.26358737e+00
 1.09268236e+00 2.45979096e+00 9.85180236e+00 3.87955679e+00
 5.42297905e+00 2.86642247e+01 2.4

## Optimize


In [54]:
# Estimate Required Parameters
import numpy as np

# Input: delta_samples (shape: [N_samples, P]), G_r_samples (shape: [N_samples])

# Estimate μδ (mean vector of δ)
mu_delta = np.mean(delta_samples, axis=0)  # Shape: [P]

# Estimate Σδ (covariance matrix of δ)
cov_delta = np.cov(delta_samples, rowvar=False)  # Shape: [P, P]

# Estimate μG(r') (mean of G^{(r')})
mu_G = np.mean(G_r_samples)

# Estimate σ²G(r') (variance of G^{(r')})
var_G = np.var(G_r_samples, ddof=1)

# Estimate σδ,G(r') (covariance vector between δ and G^{(r')})
cov_delta_G = np.array([
    np.cov(delta_samples[:, i], G_r_samples, ddof=1)[0, 1]
    for i in range(delta_samples.shape[1])
])  # Shape: [P]

In [55]:
# Check Feasibility Condition
# Example feasibility check (pseudocode)
def exists_feasible_w_with_non_negative_muY(W, mu_delta, mu_G):
    for w in W:
        muY = np.dot(w, mu_delta) - mu_G
        if muY >= 0:
            return True
    return False

has_feasible_w = exists_feasible_w_with_non_negative_muY(Wo_, mu_delta, mu_G)
has_feasible_w

True

In [44]:
# # Solve Optimization for Each λ
# from scipy.optimize import minimize
# Λ = np.array([0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.10])


# # Define the objective function
# def compute_Y_stats(w, mu_delta, cov_delta, mu_G, var_G, cov_delta_G):
#     muY = np.dot(w, mu_delta) - mu_G
#     sigmaY_sq = (
#         w.T @ cov_delta @ w  # w^T Σδ w
#         + var_G              # σ²G(r')
#         - 2 * np.dot(w, cov_delta_G)  # -2 w^T Cov(δ, G(r'))
#     )
#     return muY, sigmaY_sq

# # For each λ, solve:
# solutions = []
# for λ in Λ:
#     if has_feasible_w:
#         # Maximize μY_w - λ * σ²Y_w
#         objective = lambda w: -(np.dot(w, mu_delta) - mu_G + λ * (
#             w.T @ cov_delta @ w + var_G - 2 * np.dot(w, cov_delta_G)
#         ))
#     else:
#         # Maximize μY_w + λ * σ²Y_w
#         objective = lambda w: -(np.dot(w, mu_delta) - mu_G - λ * (
#             w.T @ cov_delta @ w + var_G - 2 * np.dot(w, cov_delta_G)
#         ))

#     # Solve with constraints (𝕎: budget, positions, etc.)
#     # This requires a MILP/MIQP solver (e.g., Gurobi, CPLEX)
#     result = solve_miqp(objective, constraints=Wo_)
#     solutions.append(result.w)

In [45]:
# conda install -c gurobi gurobi

In [46]:
# from gurobipy import *

In [34]:
from gurobipy import *
import numpy as np

# Assuming you have the following variables defined:
# - mu_delta: mean of delta
# - cov_delta: covariance matrix of delta
# - mu_G: mean of G(r')
# - var_G: variance of G(r')
# - cov_delta_G: covariance between delta and G(r')
# - Wo_: constraints (e.g., budget, position limits)

# Solve Optimization for Each λ
Λ = np.array([0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.10])

gk_indexes = [player_ids.index(gk_id) for gk_id in m_ids if gk_id in player_ids]
d_indexes = [player_ids.index(d_id) for d_id in m_ids if d_id in player_ids]
m_indexes = [player_ids.index(m_id) for m_id in m_ids if m_id in player_ids]
f_indexes = [player_ids.index(f_id) for f_id in m_ids if f_id in player_ids]

solutions = []
for λ in Λ:
    # Create a Gurobi model
    model = Model("Portfolio_Optimization")

    # Define the number of assets
    n_assets = len(mu_delta)

    # Define the portfolio weight variables
    w = model.addMVar(shape=n_assets, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS)

    # Define the objective function
    # Utility = muY - λ * sigmaY
    # Where:
    # muY = w.T @ mu_delta - mu_G
    # sigmaY = w.T @ cov_delta @ w + var_G - 2 * w.T @ cov_delta_G

    # Compute muY
    # muY = np.dot(w, mu_delta) - mu_G
    muY = w @ mu_delta - mu_G # Ensures proper handling

    # Compute sigmaY squared
    # print(cov_delta_G)
    sigmaY_sq = w.T @ cov_delta @ w + var_G - 2 *quicksum(np.dot(w, cov_delta_G))

    # Utility function: muY - λ * sigmaY_sq
    # utility = muY - λ * sigmaY_sq
    if muY >= 0:
        # If muY is non-negative, maximize the utility function
        model.setObjective(muY - λ * sigmaY_sq, GRB.MAXIMIZE)
    else:
        # If muY is negative, minimize the utility function
        model.setObjective(muY + λ * sigmaY_sq, GRB.MAXIMIZE)
    # # Set the objective to maximize utility
    # model.setObjective(quicksum(muY - λ * sigmaY_sq), GRB.MAXIMIZE)

    # Add constraints (e.g., budget constraint: sum of weights = 1)
    # Example constraint: sum(w) = 1
    w = model.addVars(len(mu_delta), vtype=GRB.BINARY, name="w")

    # Total players constraint
    model.addConstr(quicksum(w[idx] for idx , _ in enumerate(player_ids)) == 11, "total_players")

    # gk_sizes = [1]
    # def_sizes = [3,4,5]
    # mid_sizes = [2,3,4,5]
    # fwd_sizes = [1,2,3]

    # GK: exactly 1
    model.addConstr(quicksum(w[i] for i in gk_indexes) == 1, "gk_exactly_1")

    # DEF: between 3 and 5
    model.addConstr(quicksum(w[i] for i in d_indexes) >= 3, "def_min_3")
    model.addConstr(quicksum(w[i] for i in d_indexes) <= 5, "def_max_5")

    # MID: between 2 and 5
    model.addConstr(quicksum(w[i] for i in m_indexes) >= 2, "mid_min_2")
    model.addConstr(quicksum(w[i] for i in m_indexes) <= 5, "mid_max_5")

    # FWD: between 1 and 3
    model.addConstr(quicksum(w[i] for i in f_indexes) >= 1, "fwd_min_1")
    model.addConstr(quicksum(w[i] for i in f_indexes) <= 3, "fwd_max_3")


    # model.addConstr(quicksum(w) == 15, name="squad_constraint")

    # Add position limits if necessary
    # for i in range(n_assets):
    #     model.addConstr(w[i] >= 0, f"min_weight_{i}")
    #     model.addConstr(w[i] <= 1, f"max_weight_{i}")

    # Solve the model
    model.optimize()

    # Check if solution is optimal
    if model.status == GRB.OPTIMAL:
        # Extract the optimal weights
        optimal_w = [w[i].x for i in range(len(mu_delta))] # w.X
        # w_values = [w[i].x for i in range(n)]
        solutions.append(optimal_w)
    else:
        print(f"No feasible solution found for λ = {λ}")
        solutions.append(None)

# After solving for all λ, you can analyze the solutions
print("Optimal weights for each λ:")
for i, λ in enumerate(Λ):
    if solutions[i] is not None:
        print(f"λ = {λ}: {solutions[i]}")
    else:
        print(f"λ = {λ}: No feasible solution")


Set parameter Username
Set parameter LicenseID to value 2646408
Academic license - for non-commercial use only - expires 2026-04-02


GurobiError: Constraint has no bool value (are you trying "lb <= expr <= ub"?)

In [ ]:
mu_delta.to

element
2.0      2.914474
4.0      0.265789
6.0      3.636404
7.0      4.798684
8.0      1.889474
           ...   
762.0    0.215789
764.0    0.000000
765.0    0.026316
770.0    0.000000
772.0    0.000000
Length: 517, dtype: float64

In [61]:
gk_indexes = [player_ids.index(gk_id) for gk_id in g_ids if gk_id in player_ids]
d_indexes = [player_ids.index(d_id) for d_id in d_ids if d_id in player_ids]
m_indexes = [player_ids.index(m_id) for m_id in m_ids if m_id in player_ids]
f_indexes = [player_ids.index(f_id) for f_id in f_ids if f_id in player_ids]

m = Model("Portfolio_Optimization")
# Define the number of assets
n_assets = len(mu_delta)

# Define the portfolio weight variables
w = m.addMVar(shape=n_assets, vtype=GRB.BINARY, name="w")

# Add constraints (e.g., budget constraint: sum of weights = 1)

# Total players constraint
m.addConstr(sum(w[idx] for idx , _ in enumerate(player_ids)) == 11, "total_players")


# Add position limits if necessary
# GK: exactly 1
m.addConstr(sum(w[i] for i in gk_indexes) == 1, "gk_exactly_1")

# DEF: between 3 and 5
m.addConstr(sum(w[i] for i in d_indexes) >= 3, "def_min_3")
m.addConstr(sum(w[i] for i in d_indexes) <= 5, "def_max_5")

# MID: between 2 and 5
m.addConstr(sum(w[i] for i in m_indexes) >= 4, "mid_min_2")
m.addConstr(sum(w[i] for i in m_indexes) <= 5, "mid_max_5")

# FWD: between 1 and 3
m.addConstr(sum(w[i] for i in f_indexes) >= 3, "fwd_min_1")
m.addConstr(sum(w[i] for i in f_indexes) <= 3, "fwd_max_3")

# Assuming x is a dictionary of binary variables: x[i] for player i
muY = sum(mu_delta[i] * w[i] for i in range(n_assets))

sigmaY_sq = sum(cov_delta[i][j] * w[i] * w[j] for i in range(n_assets) for j in range(n_assets))

# Set the objective function
m.setObjective(muY - 0.5 * sigmaY_sq, GRB.MAXIMIZE)
# Solve the model

In [62]:
# Print variables and constraints before solving
print("\n🔍 Variable Info:")
for i in range(n_assets):
    print(f"w[{i}] ({player_ids[i]}): Binary")

print("\n📏 Constraints:")
for c in m.getConstrs():
    print(f"{c.ConstrName}")

# Solve the model
m.optimize()

# Output solution
if m.status == GRB.OPTIMAL:
    print("\n✅ Selected Players:")
    selected = [i for i in range(n_assets) if w[i].X > 0.5]
    for i in selected:
        print(f"- {player_ids[i]} (index {i}, expected points = {mu_delta[i]})")
else:
    print("\n❌ No feasible solution found.")


🔍 Variable Info:
w[0] (10.0): Binary
w[1] (100.0): Binary
w[2] (101.0): Binary
w[3] (102.0): Binary
w[4] (104.0): Binary
w[5] (105.0): Binary
w[6] (106.0): Binary
w[7] (107.0): Binary
w[8] (109.0): Binary
w[9] (11.0): Binary
w[10] (110.0): Binary
w[11] (111.0): Binary
w[12] (113.0): Binary
w[13] (115.0): Binary
w[14] (117.0): Binary
w[15] (12.0): Binary
w[16] (120.0): Binary
w[17] (121.0): Binary
w[18] (122.0): Binary
w[19] (123.0): Binary
w[20] (124.0): Binary
w[21] (126.0): Binary
w[22] (128.0): Binary
w[23] (129.0): Binary
w[24] (13.0): Binary
w[25] (131.0): Binary
w[26] (132.0): Binary
w[27] (134.0): Binary
w[28] (135.0): Binary
w[29] (136.0): Binary
w[30] (137.0): Binary
w[31] (14.0): Binary
w[32] (141.0): Binary
w[33] (142.0): Binary
w[34] (144.0): Binary
w[35] (145.0): Binary
w[36] (146.0): Binary
w[37] (147.0): Binary
w[38] (148.0): Binary
w[39] (149.0): Binary
w[40] (15.0): Binary
w[41] (152.0): Binary
w[42] (153.0): Binary
w[43] (156.0): Binary
w[44] (157.0): Binary
w[45] (1

In [63]:
selected

[9, 10, 15, 16, 231, 234, 247, 248, 250, 266, 272]

In [ ]:
from gurobipy import *
import numpy as np

best_lambda = None
best_w = None
max_prob = -float('inf')

# feasible_muY_positive = any(muY_w >= 0 for w in Wo_)

lambda_values = np.arange(0, 2.05, 0.05)  # From 0 to 2, in steps of 0.05


gk_indexes = [player_ids.index(gk_id) for gk_id in m_ids if gk_id in player_ids]
d_indexes = [player_ids.index(d_id) for d_id in m_ids if d_id in player_ids]
m_indexes = [player_ids.index(m_id) for m_id in m_ids if m_id in player_ids]
f_indexes = [player_ids.index(f_id) for f_id in m_ids if f_id in player_ids]
for λ in lambda_values:
    model = Model("Portfolio_Optimization")

    n_assets = len(mu_delta)

    # Define the portfolio weight variables
    w = model.addMVar(shape=n_assets, lb=0.0, ub=1.0, vtype=GRB.BINARY, name="w")

    # # Define binary variables for player selection: w[i] ∈ {0,1}
    # w = model.addVars(len(player_ids), vtype=GRB.BINARY, name="w")

    # Total players constraint
    model.addConstr(quicksum(w[idx] for idx , _ in enumerate(player_ids)) == 11, "total_players")

    # gk_sizes = [1]
    # def_sizes = [3,4,5]
    # mid_sizes = [2,3,4,5]
    # fwd_sizes = [1,2,3]

    # GK: exactly 1
    model.addConstr(quicksum(w[i] for i in gk_indexes) == 1, "gk_exactly_1")

    # DEF: between 3 and 5
    model.addConstr(quicksum(w[i] for i in d_indexes) >= 3, "def_min_3")
    model.addConstr(quicksum(w[i] for i in d_indexes) <= 5, "def_max_5")

    # MID: between 2 and 5
    model.addConstr(quicksum(w[i] for i in m_indexes) >= 2, "mid_min_2")
    model.addConstr(quicksum(w[i] for i in m_indexes) <= 5, "mid_max_5")

    # FWD: between 1 and 3
    model.addConstr(quicksum(w[i] for i in f_indexes) >= 1, "fwd_min_1")
    model.addConstr(quicksum(w[i] for i in f_indexes) <= 3, "fwd_max_3")

    # # muY = sum(predicted_points[i] * w[i])
    # muY = quicksum(muY_vals[i] * w[i] for idx , _ in enumerate(player_ids))
    # sigmaY_sq = quicksum(sigmaY_sq[i] * w[i] for idx , _ in enumerate(player_ids))

    # # Compute muY
    # # muY = np.dot(w, mu_delta) - mu_G
    # muY = w @ mu_delta - mu_G # Ensures proper handling

    # # Compute sigmaY squared
    # # print(cov_delta_G)
    # sigmaY_sq = w.T @ cov_delta @ w + var_G - 2 *quicksum(np.dot(w, cov_delta_G))
    print(model)
    for i in range(len(w)):
        print(f"w[{i}]: Name={w[i].VarName}, LB={w[i].LB}, UB={w[i].UB}, VType={w[i].VType}")

    # # Assuming x is a dictionary of binary variables: x[i] for player i
    # muY = quicksum(mu_delta[i] * w[i] for i in range(n_assets))

    # sigmaY_sq = quicksum(
    #     cov_delta[i][j] * w[i] * w[j]
    #     for i in range(n_assets) for j in range(n_assets)
    # )



    # if has_feasible_w:
    #     # Add constraint: muY ≥ 0
    #     model.addConstr(muY >= 0)
    #     model.setObjective(muY - λ * sigmaY_sq, GRB.MAXIMIZE)
    # else:
    #     model.setObjective(muY + λ * sigmaY_sq, GRB.MAXIMIZE)

    # model.optimize()

    # if model.status == GRB.OPTIMAL:
    #     selected_w = model.getAttr("X", w)
    #     muY_val = quicksum(muY[i] * selected_w[i] for idx , _ in enumerate(player_ids))
    #     sigmaY_val = quicksum(sigmaY_sq[i] * selected_w[i] for idx , _ in enumerate(player_ids)) # ** 0.5

    #     # Compute probability that Y_w > 0 using normal approximation
    #     prob = 1 - norm.cdf(0, loc=muY_val, scale=sigmaY_val)

    #     if prob > max_prob:
    #         max_prob = prob
    #         best_lambda = λ
    #         best_w = selected_w


<gurobi.Model Continuous instance Portfolio_Optimization: 0 constrs, 0 vars, Parameter changes: Username=(user-defined), LicenseID=2646408>


TypeError: object of type 'MVar' has no len()

In [ ]:
best_lambda ,best_w ,max_prob

(None, None, -inf)

In [ ]:
np.sum(solutions[0].tolist())

1.0000000000000002